# How Linear Is a Transformer's Feed-Forward Block?

Companion notebook for the [blog post](https://Notabot123.github.io/polyweave/blog/how-linear-is-a-transformer-ffn/).
The first demo is self-contained; the GPT-2 cell downloads the model and needs the
`[distill]` extra (`transformers`).

In [ ]:
import torch
import torch.nn as nn
from polyweave.distill import fit_closed_form_linear, IOCapture

## A toy block (self-contained)

Linear map plus a genuinely nonlinear residual `(X**2 - 1)`, which no linear map can absorb.

In [ ]:
torch.manual_seed(0)
d, N = 64, 8_000
X = torch.randn(N, d)
W_true = torch.randn(d, d) / d**0.5
M2 = torch.randn(d, d) / d**0.5
Y = X @ W_true.T + 0.25 * (X**2 - 1.0) @ M2

result = fit_closed_form_linear(nn.Linear(d, d), X, Y)
print(f"R2_lin = {result.val_r2:.3f}")   # ~0.882: mostly linear, ~12% genuine residual

## A real GPT-2 block

Capture an FFN block's input/output activation pairs with `IOCapture`, then fit the exact
closed-form linear map. **This downloads GPT-2 (~500 MB) on first run.** `transformers` is
imported here (not at the top) so the toy demo above runs without the `[distill]` extra.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
tok = AutoTokenizer.from_pretrained("gpt2")

# A small corpus just to make the cell run. For the paper's ~0.95 you need a real corpus
# (e.g. WikiText-2, thousands of tokens > the 768 feature dim); see the note below.
corpus = [
    "The early morning fog rolled across the quiet harbour as the boats returned.",
    "Mathematics is the language in which the laws of nature are most clearly written.",
    "She closed the heavy book and stared out of the rain-streaked window.",
    "Each layer of the network transforms its input before passing it onward.",
    "History rarely repeats itself exactly, but its rhythms are often familiar.",
    "The committee debated the proposal for hours before reaching a compromise.",
    "A single well-chosen example can illuminate an idea better than a long proof.",
    "Migrating birds navigate using a remarkable combination of cues and instinct.",
]

block = model.transformer.h[1].mlp                  # GPT-2's early FFN sub-block
with IOCapture(block, max_rows=4_000) as cap:
    for text in corpus:
        ids = tok(text, return_tensors="pt").input_ids[:, :128]
        with torch.no_grad():
            model(ids)                              # forward passes drive the hook
X, Y = cap.pairs()                                  # [N, 768], [N, 768]
print("captured rows:", X.shape[0])
print("R2_lin =", round(fit_closed_form_linear(nn.Linear(X.shape[1], Y.shape[1]), X, Y).val_r2, 3))

> **Full study.** This tiny corpus gives only a rough number (few rows vs the 768-d block).
> For the paper's multi-model, multi-block measurement (the real ~0.95 and the depth
> curves), run `python -m polyweave.experiments.gpt2_mlp_distill`.